# Практика 17 · Перенавчання, недонавчання і розклад помилки

> 📖 **Теорія:** відкрий `lecture.html` у цій же теці.
> 📝 **Домашнє завдання:** `homework.md`. 🧠 **Тест:** `quiz.html`.

Лекція показала дві протилежні невдачі моделі й пояснила, звідки береться
U-подібна крива. Тут ми відтворимо обидві хвороби своїми руками, а потім
розкладемо помилку на три доданки й перевіримо, що вони справді сумуються.

**Що зробимо:**
1. Породимо ціни телефонів із **відомої** нам залежності плюс шум
2. Навчимо поліноми різних степенів і перевіримо свою реалізацію бібліотечною
3. Побудуємо таблицю «степінь → помилка на навчанні → помилка на тесті»
4. Знайдемо **точку перелому** — момент, з якого починається перенавчання
5. Повторимо все на вдесятеро більшій вибірці й побачимо, що перелом зсунувся
6. Подивимось на недонавчання в чистому вигляді
7. Побудуємо **сорок паралельних дощок** і побачимо зміщення й дисперсію очима
8. Порахуємо три доданки числом і перевіримо тотожність розкладу
9. Оцінимо дисперсію **бутстрепом**, коли дошка одна-єдина
10. Побудуємо ті самі криві готовими `validation_curve` і `learning_curve`

> Числа тут не збігатимуться з лекцією до гривні: там дані породжував генератор
> браузера, тут — NumPy. Явище й усі висновки ті самі.

## 0. Світ, у якому ми знаємо істину

Уявімо дошку оголошень про вживані телефони рідкісної марки. Ціна залежить від
року випуску: телефон дешевшає з віком, але не по прямій — свіжий флагман втрачає
тисячі гривень за рік, а десятирічний апарат уже майже не дешевшає. Плюс невеликий
горб на 2019-му: тодішня серія вийшла вдалою й тримає ціну краще за сусідні роки.

До цієї закономірності додається **шум** — усе, що не пояснюється роком: подряпина
на корпусі, продавець поспішає, продавець поставив із запасом.

У житті істину ніхто не знає. Тут знаємо ми, бо самі її задали, — і саме тому
зможемо показати пальцем, що модель вивчила закономірність, а що шум.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import legendre

FIRST_YEAR = 2010          # найстаріший телефон на дошці
LAST_YEAR = 2025           # найновіший
PRICE_NOISE = 750          # шум: стандартне відхилення в гривнях


def true_price(year):
    """Закономірність, якої модель не знає: здешевлення з віком плюс горб на 2019."""
    depreciation = 1900 + 18500 * 0.80 ** (LAST_YEAR - year)
    good_series = 1500 * np.exp(-((year - 2019) / 1.3) ** 2)
    return depreciation + good_series


def draw_board(rng, how_many):
    """Одна дошка оголошень: справжня ціна свого року плюс випадкове відхилення.

    Роки розкидаємо не як завгодно, а по одному з кожного рівного відрізка часу:
    саме так виглядає реальна дошка, де щороку зʼявляється кілька оголошень.
    Заодно це рятує підгонку — при випадкових роках два оголошення легко
    опиняються поруч, матриця ознак стає майже виродженою, і на високих степенях
    ми поміряли б похибку арифметики замість дисперсії моделі.
    """
    slots = np.arange(how_many)
    span = LAST_YEAR - FIRST_YEAR
    years = FIRST_YEAR + (slots + 0.14 + 0.72 * rng.random(how_many)) / how_many * span
    prices = true_price(years) + rng.normal(0, PRICE_NOISE, how_many)
    return years, prices


rng = np.random.default_rng(42)
train_years, train_prices = draw_board(rng, 16)
test_years, test_prices = draw_board(rng, 300)

print(f"навчальних оголошень: {len(train_years)}")
print(f"тестових оголошень:   {len(test_years)}")
print("\nперші пʼять навчальних оголошень:")
for year, price in zip(train_years[:5], train_prices[:5]):
    print(f"  {year:.2f} року — {price:8.0f} грн   "
          f"(типова ціна цього року: {true_price(year):.0f} грн)")

## 1. Модель: поліном, але обережно

Складність моделі регулює **степінь полінома**. Тут є технічна пастка, про яку
варто знати заздалегідь.

Якщо будувати ознаки як сирі степені року — `[1, рік, рік², …, рік¹⁵]` — то при
роках близько 2020 значення `рік¹⁵` має порядок 10⁴⁹. Стовпці матриці стають
майже однаковими, система погано обумовлена, і на високих степенях ми отримаємо
не перенавчання, а **чисельне сміття**.

Рятує це дві дії разом: спершу стиснути роки у відрізок `[-1, 1]`, потім узяти не
сирі степені, а **поліноми Лежандра** — вони задають той самий простір функцій, але
їхні стовпці майже ортогональні.

In [ ]:
def to_unit(years):
    """Стискаємо роки в [-1, 1]: без цього високі степені розвалюються чисельно."""
    return 2 * (years - FIRST_YEAR) / (LAST_YEAR - FIRST_YEAR) - 1


def fit_polynomial(years, prices, degree):
    """МНК-поліном заданого степеня. Повертає коефіцієнти в базисі Лежандра."""
    design = legendre.legvander(to_unit(years), degree)
    coefficients, *_ = np.linalg.lstsq(design, prices, rcond=None)
    return coefficients


def predict(coefficients, years):
    """Ціна, яку модель називає для кожного року."""
    degree = len(coefficients) - 1
    return legendre.legvander(to_unit(years), degree) @ coefficients


coefficients_3 = fit_polynomial(train_years, train_prices, 3)
print("коефіцієнти полінома 3-го степеня:", np.round(coefficients_3, 1))
print("прогноз для телефона 2020 року:",
      round(float(predict(coefficients_3, np.array([2020.0]))[0])), "грн")

Перевіримо, що всередині немає магії: та сама задача через `scikit-learn`, тільки
на звичайних степенях. Простір функцій той самий — отже, прогноз має збігтися.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

library_model = make_pipeline(PolynomialFeatures(3), LinearRegression())
library_model.fit(to_unit(train_years).reshape(-1, 1), train_prices)

our_prediction = predict(coefficients_3, test_years)
library_prediction = library_model.predict(to_unit(test_years).reshape(-1, 1))

print(f"найбільше розходження: {np.abs(our_prediction - library_prediction).max():.2e} грн")

assert np.allclose(our_prediction, library_prediction), "розрахунок розійшовся!"
print("\n✅ збігається — усередині бібліотеки той самий МНК")

## 2. Помилка: одна формула, дві вибірки

Міряти будемо **RMSE** — корінь із середнього квадрата помилки. Зручність у тому,
що результат виходить у гривнях: «модель у середньому промахується на стільки-то».

Головне тут не формула, а те, що ми рахуємо її **двічі**: на тих оголошеннях, за
якими вчились, і на тих, яких модель не бачила.

In [ ]:
def rmse(actual, predicted):
    """Середня квадратична помилка в гривнях."""
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def two_errors(degree, years, prices):
    """Помилка тієї самої моделі на навчальних і на тестових оголошеннях."""
    coefficients = fit_polynomial(years, prices, degree)
    on_train = rmse(prices, predict(coefficients, years))
    on_test = rmse(test_prices, predict(coefficients, test_years))
    return on_train, on_test


for degree in (1, 3, 9, 15):
    train_error, test_error = two_errors(degree, train_years, train_prices)
    print(f"степінь {degree:>2}:  навчання {train_error:9.0f} грн   тест {test_error:12.0f} грн")

Уже видно обидві хвороби. При степені 1 обидва числа високі й майже однакові —
моделі бракує гнучкості. При степені 15 навчальна помилка практично нульова
(коефіцієнтів рівно стільки ж, скільки оголошень, тож крива проходить точно через
кожне), а тестова — астрономічна.

## 3. Уся таблиця степенів

Тепер порахуємо це для всіх степенів підряд і складемо в таблицю.

In [ ]:
import pandas as pd

rows = []
for degree in range(1, 16):
    train_error, test_error = two_errors(degree, train_years, train_prices)
    rows.append({"степінь": degree,
                 "помилка на навчанні": round(train_error),
                 "помилка на тесті": round(test_error),
                 "розрив": round(test_error - train_error)})

table = pd.DataFrame(rows).set_index("степінь")
print(table.to_string())

## 4. Точка перелому

Читати таблицю очима незручно, тому знайдемо перелом програмно. Нас цікавлять
дві речі:

- **найкращий степінь** — той, де тестова помилка найменша;
- **перший степінь, після якого тестова помилка пішла вгору, а навчальна далі
  падає.** Це і є момент переходу в перенавчання.

In [ ]:
train_errors = table["помилка на навчанні"].to_numpy()
test_errors = table["помилка на тесті"].to_numpy()
degrees_all = table.index.to_numpy()

best_degree = int(degrees_all[test_errors.argmin()])
best_error = int(test_errors.min())

print(f"найкращий степінь: {best_degree}")
print(f"його помилка на тесті: {best_error} грн")
print(f"його помилка на навчанні: {train_errors[test_errors.argmin()]} грн")
print()

# перевіряємо, що навчальна помилка справді ніколи не росте
always_falls = np.all(np.diff(train_errors) <= 0)
print(f"навчальна помилка ніколи не росте: {always_falls}")
print(f"а тестова після степеня {best_degree} — росте у "
      f"{int(np.sum(np.diff(test_errors[best_degree - 1:]) > 0))} випадках із "
      f"{len(test_errors) - best_degree}")

Тепер намалюємо те, заради чого все й затівалось: дві криві помилки.
Шкала помилки логарифмічна — інакше степінь 15 розчавив би всю решту в лінію.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(degrees_all, train_errors, "o-", color="#0f766e", label="помилка на навчанні")
ax.plot(degrees_all, test_errors, "o-", color="#c2185b", label="помилка на тесті")
ax.axvline(best_degree, color="#555", linestyle="--", linewidth=1)
ax.set_yscale("log")
ax.set_xlabel("степінь полінома")
ax.set_ylabel("RMSE, грн  (логарифмічна шкала)")
ax.set_title("Навчальна помилка падає завжди, тестова має форму літери U")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"пунктир — найкращий степінь ({best_degree})")

І три моделі поруч, щоб побачити ті самі числа очима: занадто проста, вдала
й перенавчена.

In [ ]:
year_grid = np.linspace(FIRST_YEAR, LAST_YEAR, 400)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, degree in zip(axes, (1, best_degree, 15)):
    coefficients = fit_polynomial(train_years, train_prices, degree)
    ax.plot(year_grid, true_price(year_grid), "--", color="#888", label="справжня залежність")
    ax.plot(year_grid, predict(coefficients, year_grid), color="#c2185b", label="модель")
    ax.scatter(train_years, train_prices, color="#17212b", zorder=3, s=25, label="оголошення")
    train_error, test_error = two_errors(degree, train_years, train_prices)
    ax.set_title(f"степінь {degree}\nнавчання {train_error:.0f} грн · тест {test_error:.0f} грн",
                 fontsize=10)
    ax.set_ylim(0, 24000)          # обрізаємо: поліном 15-го степеня вилітає на мільйони
    ax.set_xlabel("рік випуску")

axes[0].set_ylabel("ціна, грн")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("зверни увагу: права крива проходить точно через кожну точку — і саме тому вона найгірша")

## 5. Та сама модель, але оголошень удесятеро більше

Складність моделі завжди відносна до обсягу даних. Поліном 9-го степеня на
16 оголошеннях — це складна модель. На 160 оголошеннях — цілком помірна.

Перевіримо: згенеруємо вдесятеро більшу дошку з того самого джерела
й побудуємо ту саму таблицю.

In [ ]:
big_years, big_prices = draw_board(rng, 160)

big_rows = []
for degree in range(1, 16):
    train_error, test_error = two_errors(degree, big_years, big_prices)
    big_rows.append({"степінь": degree,
                     "помилка на навчанні": round(train_error),
                     "помилка на тесті": round(test_error),
                     "розрив": round(test_error - train_error)})

big_table = pd.DataFrame(big_rows).set_index("степінь")
print(big_table.to_string())

In [ ]:
best_on_16 = best_degree
best_on_160 = int(big_table["помилка на тесті"].idxmin())

print(f"на  16 оголошеннях найкращий степінь: {best_on_16}")
print(f"на 160 оголошеннях найкращий степінь: {best_on_160}")
print()

for degree in (9, 15):
    gap_16 = table.loc[degree, "розрив"]
    gap_160 = big_table.loc[degree, "розрив"]
    print(f"степінь {degree:>2}: розрив на 16 оголошеннях {gap_16:>9} грн, "
          f"на 160 — {gap_160:>6} грн")

print("\nмодель не змінилась ані на коефіцієнт — змінилось те, скільки їй довелось вигадувати")

## 6. Недонавчання в чистому вигляді

І контрольний дослід. Якщо перенавчання лікується даними, то, може, дані вилікують
і недонавчання? Візьмемо пряму (степінь 1) і будемо давати їй усе більше й більше
оголошень.

In [ ]:
sizes = [16, 40, 100, 400, 2000]
print(f"{'оголошень':>10} | {'навчання':>12} | {'тест':>12}")
print("-" * 40)
for size in sizes:
    some_years, some_prices = draw_board(rng, size)
    train_error, test_error = two_errors(1, some_years, some_prices)
    print(f"{size:>10} | {train_error:>9.0f} грн | {test_error:>9.0f} грн")

print("\nдві тисячі оголошень — і жодного покращення: пряма лишається прямою")

Порівняй це з попереднім розділом. Там дані **різко** зменшили розрив між помилками.
Тут вони не змінили нічого, бо проблема не в розриві: обидві помилки високі й
однакові з самого початку.

Ось і вся діагностика в одному рядку: **дивись не на одну помилку, а на дві —
і на відстань між ними.**

---

## 7. Сорок паралельних дощок

Досі ми дивились на симптоми. Тепер розберемо помилку на частини.

Головна ідея розділу 10 лекції: зміщення й дисперсія — властивості **процедури
навчання**, а не однієї навченої моделі. Побачити їх на одному прогоні неможливо.

Тому влаштовуємо уявний експеримент по-справжньому: витягуємо багато різних дощок
оголошень з того самого джерела, на кожній навчаємо свою модель і дивимось на
розкид прогнозів.

In [ ]:
N_BOARDS = 200          # скільки паралельних дощок проживаємо
BOARD_SIZE = 16         # стільки ж оголошень, скільки в нашій єдиній дошці

# сітка років, на якій міряємо: краї відрізані, бо там будь-який поліном шаліє
# і числа перестають щось означати
grid_years = np.linspace(2011, 2024, 131)
grid_truth = true_price(grid_years)


def run_parallel_boards(degree, size=BOARD_SIZE, boards=N_BOARDS, seed=0):
    """Навчає `boards` моделей на різних дошках. Повертає матрицю прогнозів.

    Рядок — одна дошка, стовпець — один рік із сітки.
    """
    board_rng = np.random.default_rng(seed)
    predictions = np.zeros((boards, len(grid_years)))
    for board in range(boards):
        years, prices = draw_board(board_rng, size)
        predictions[board] = predict(fit_polynomial(years, prices, degree), grid_years)
    return predictions


predictions_degree_3 = run_parallel_boards(degree=3)
middle = len(grid_years) // 2
print(f"матриця прогнозів: {predictions_degree_3.shape} (дощок × років сітки)")
print(f"для {grid_years[middle]:.1f} року прогнози гуляють від "
      f"{predictions_degree_3[:, middle].min():.0f} до "
      f"{predictions_degree_3[:, middle].max():.0f} грн")
print(f"справжня ціна там: {grid_truth[middle]:.0f} грн")

### Віяло

Кожна тонка лінія — модель з окремої дошки. Товста рожева — **середня модель**
$\bar{f}$, та сама, що стоїть у визначенні зміщення. Сірий пунктир — істина.

Дивись на дві речі **окремо**:
- наскільки товста лінія розходиться з пунктиром — це **зміщення**;
- наскільки широке віяло — це **дисперсія**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)

for ax, degree in zip(axes, [1, 5, 12]):
    predictions = run_parallel_boards(degree)
    # малюємо лише 40 дощок — інакше картинка перетвориться на суцільну пляму
    for board in range(40):
        ax.plot(grid_years, predictions[board], color="teal", alpha=.16, lw=1)
    ax.plot(grid_years, predictions.mean(axis=0), color="crimson", lw=3, label="середня модель")
    ax.plot(grid_years, grid_truth, color="grey", ls="--", lw=2, label="істина")
    ax.set_title(f"степінь {degree}")
    ax.set_xlabel("рік випуску")
    ax.grid(alpha=.25)

axes[0].set_ylabel("ціна, грн")
axes[0].set_ylim(0, 24000)
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

print("Степінь 1: усі прямі лежать майже одна на одній (дисперсії немає),")
print("           але жодна навіть не намагається повторити горб 2019 року.")
print("Степінь 12: середня крива лягла на істину чудово, зате окрема модель")
print("           може дати будь-що. Це і є компроміс, буквально очима.")

## 8. Рахуємо три доданки числом

Формули з лекції, слово в слово:

$$\text{Зміщення}(x) = \mathbb{E}_D[\hat{f}_D(x)] - f(x), \qquad
\text{Дисперсія}(x) = \mathbb{E}_D\big[(\hat{f}_D(x) - \mathbb{E}_D[\hat{f}_D(x)])^2\big]$$

Сподівання $\mathbb{E}_D$ береться **по дошках** — тобто по рядках нашої матриці
прогнозів. Потім усереднюємо по роках сітки, щоб отримати одне число на модель.

Величини виводимо в гривнях (корінь із доданка), щоб їх можна було порівняти
з RMSE з першої половини зошита.

In [ ]:
def decompose(predictions):
    """Розкладає помилку на три доданки. Повертає (зміщення², дисперсія, шум)."""
    average_model = predictions.mean(axis=0)                  # f̄(x): середнє по дошках
    bias_squared = np.mean((average_model - grid_truth) ** 2)  # промах середньої моделі
    variance = np.mean(predictions.var(axis=0))                # розліт навколо середньої
    noise = PRICE_NOISE ** 2                                   # підлога, однакова завжди
    return bias_squared, variance, noise


print(f"{'степінь':>8} {'зміщення':>11} {'дисперсія':>11} {'шум':>8} {'разом':>9}")
for degree in [1, 3, 5, 8, 12]:
    bias_squared, variance, noise = decompose(run_parallel_boards(degree))
    print(f"{degree:>8} {np.sqrt(bias_squared):>11.0f} {np.sqrt(variance):>11.0f} "
          f"{np.sqrt(noise):>8.0f} {np.sqrt(bias_squared + variance + noise):>9.0f}")
print("\nусі числа — у гривнях; складаються вони в квадратах, тому «разом» —")
print("це корінь із суми квадратів, а не сума стовпців")

### Перевірка 1: доданки справді сумуються

Лекція наполягає, що розклад — **тотожність**, а не наближення. Перевіримо це
буквально: порахуємо середній квадрат відхилення прогнозів від істини напряму,
без жодного розкладу, і порівняємо зі сумою зміщення² та дисперсії.

In [ ]:
predictions = run_parallel_boards(degree=5)
bias_squared, variance, noise = decompose(predictions)

# «в лоб»: середній по дошках і по роках квадрат відхилення прогнозу від істини
straight_error = np.mean((predictions - grid_truth) ** 2)

print(f"зміщення² + дисперсія = {bias_squared + variance:.6f}")
print(f"пораховано напряму    = {straight_error:.6f}")

assert np.allclose(bias_squared + variance, straight_error), "розклад не сходиться!"
print("\n✅ тотожність підтверджена: доданків рівно два плюс шум, і вони не перекриваються")

### Перевірка 2: а тепер із справжнім шумом

Попередня перевірка була алгебраїчною: істину ми знали точно. Тепер зробимо
чесніше — у кожній дошці згенеруємо **нові зашумлені** ціни для років сітки, як
у реальному житті, і поміряємо помилку на них.

Тут уже працює статистика, тому точного збігу не буде — лише збіг у межах
похибки Монте-Карло. Саме так і має бути.

In [ ]:
noise_rng = np.random.default_rng(123)

# у кожній дошці — свій свіжий шум на роках сітки
noisy_prices = grid_truth + noise_rng.normal(0, PRICE_NOISE, predictions.shape)
measured_error = np.mean((noisy_prices - predictions) ** 2)

predicted_by_decomposition = bias_squared + variance + noise

print(f"поміряна помилка на зашумлених цінах: {np.sqrt(measured_error):.0f} грн")
print(f"передбачення розкладу:                {np.sqrt(predicted_by_decomposition):.0f} грн")
print(f"розбіжність: "
      f"{abs(measured_error - predicted_by_decomposition) / predicted_by_decomposition * 100:.2f}%")

assert np.allclose(measured_error, predicted_by_decomposition, rtol=0.05), \
    "розклад не описує реальну помилку!"
print("\n✅ розклад передбачає реальну помилку з точністю до похибки Монте-Карло")

## 9. Розклад по складності — головна картинка теми

Ті самі числа, але для всіх степенів одразу. Сірий фундамент однаковий скрізь:
це шум, підлога, нижче якої не опуститься ніхто.

In [ ]:
degrees = np.arange(1, 13)
bias_by_degree = np.zeros(len(degrees))
variance_by_degree = np.zeros(len(degrees))

for i, degree in enumerate(degrees):
    bias_by_degree[i], variance_by_degree[i], _ = decompose(run_parallel_boards(degree))

total_by_degree = bias_by_degree + variance_by_degree + PRICE_NOISE ** 2
best_by_decomposition = degrees[total_by_degree.argmin()]

print(f"{'степінь':>8} {'зміщення':>11} {'дисперсія':>11} {'разом':>9}")
for i, degree in enumerate(degrees):
    mark = "  ← мінімум" if degree == best_by_decomposition else ""
    print(f"{degree:>8} {np.sqrt(bias_by_degree[i]):>11.0f} "
          f"{np.sqrt(variance_by_degree[i]):>11.0f} "
          f"{np.sqrt(total_by_degree[i]):>9.0f}{mark}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

noise_row = np.full(len(degrees), PRICE_NOISE ** 2)
axes[0].bar(degrees, noise_row, color="lightgrey", label="шум σ²")
axes[0].bar(degrees, bias_by_degree, bottom=noise_row, color="crimson", label="зміщення²")
axes[0].bar(degrees, variance_by_degree, bottom=noise_row + bias_by_degree,
            color="teal", label="дисперсія")
axes[0].set_yscale("log")
axes[0].set_ylabel("внесок у помилку, грн² (лог. шкала)")
axes[0].set_title("З чого складається помилка")

axes[1].plot(degrees, np.sqrt(bias_by_degree), "o-", color="crimson", lw=2, label="зміщення")
axes[1].plot(degrees, np.sqrt(variance_by_degree), "o-", color="teal", lw=2, label="дисперсія")
axes[1].plot(degrees, np.sqrt(total_by_degree), "o-", color="black", lw=2.5, label="разом")
axes[1].axvline(best_by_decomposition, ls="--", color="grey",
                label=f"мінімум суми: {best_by_decomposition}")
axes[1].set_yscale("log")
axes[1].set_ylabel("грн (лог. шкала)")
axes[1].set_title("Зміщення падає, дисперсія росте")

for ax in axes:
    ax.set_xlabel("степінь полінома")
    ax.legend()
    ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

print(f"Мінімум суми — на степені {best_by_decomposition}. Зверни увагу: це не там, де")
print("найменше зміщення, і не там, де найменша дисперсія. Оптимізувати треба суму.")
print()
if best_by_decomposition == best_degree:
    print(f"На одній-єдиній дошці з розділу 4 переможцем теж вийшов степінь {best_degree}.")
    print("Пощастило: одна дошка не зобовʼязана давати ту саму відповідь, що й двісті.")
else:
    print(f"А на одній-єдиній дошці з розділу 4 переможцем був степінь {best_degree}.")
    print("Розбіжність не помилка: там ми міряли одну дошку, тут — середнє по двохстах.")
print("Саме про це розділ 09 лекції: вибір за однією вибіркою систематично оптимістичний.")

## 10. Бутстреп: те саме, коли дошка одна

Паралельні дошки — уявний експеримент. Але дещо схоже можна зробити й насправді:
**бутстреп**. Беремо одну наявну дошку й багато разів витягуємо з неї
$n$ оголошень **з поверненням**. Кожна така підвибірка трохи інша — от і
«паралельні дошки», зроблені з підручних матеріалів.

Бутстреп не знає істини, тому зміщення він оцінити не може. А от **дисперсію** —
цілком: вона не потребує знання $f$.

In [ ]:
def bootstrap_predictions(years, prices, degree, resamples=N_BOARDS, seed=5):
    """Багато моделей, навчених на підвибірках з поверненням з однієї дошки."""
    boot_rng = np.random.default_rng(seed)
    size = len(years)
    predictions = np.zeros((resamples, len(grid_years)))
    for i in range(resamples):
        # витягуємо номери рядків з поверненням — частина оголошень повториться,
        # частина не потрапить зовсім, і саме це створює різницю між моделями
        chosen = boot_rng.integers(0, size, size)
        predictions[i] = predict(
            fit_polynomial(years[chosen], prices[chosen], degree), grid_years)
    return predictions


print(f"{'степінь':>8} {'дисперсія (200 дощок)':>23} {'дисперсія (бутстреп)':>23} {'відношення':>12}")
for degree in [1, 3, 5]:
    _, ideal_variance, _ = decompose(run_parallel_boards(degree))
    bootstrap_variance = np.mean(
        bootstrap_predictions(train_years, train_prices, degree).var(axis=0))
    print(f"{degree:>8} {np.sqrt(ideal_variance):>20.0f} грн "
          f"{np.sqrt(bootstrap_variance):>20.0f} грн "
          f"{np.sqrt(bootstrap_variance / ideal_variance):>12.2f}")

print("\nНапрямок бутстреп ловить правильно: зі складністю дисперсія росте.")
print("А от абсолютні числа він завищує, і тим сильніше, чим складніша модель.")
print("Причина: підвибірка з поверненням містить у середньому лише 63% різних")
print("оголошень. Гнучкому поліному цього критично мало — він хапається за дублікати,")
print("і його розкид роздувається. Тому бутстреп — індикатор, а не вимірювальний прилад.")

## 11. Дані бʼють лише по дисперсії

Лекція: $\text{Дисперсія} \approx C/n$, а зміщення від $n$ **не залежить взагалі**.
Перевіримо обидва твердження одразу, зафіксувавши складність.

In [ ]:
board_sizes = np.array([20, 40, 80, 160, 320])
bias_by_size = np.zeros(len(board_sizes))
variance_by_size = np.zeros(len(board_sizes))

for i, size in enumerate(board_sizes):
    predictions = run_parallel_boards(degree=5, size=int(size), boards=120)
    bias_by_size[i], variance_by_size[i], _ = decompose(predictions)

print(f"{'n':>6} {'зміщення':>11} {'дисперсія':>11} {'дисперсія × n':>16}")
for i, size in enumerate(board_sizes):
    print(f"{size:>6} {np.sqrt(bias_by_size[i]):>11.0f} {np.sqrt(variance_by_size[i]):>11.0f} "
          f"{variance_by_size[i] * size:>16.0f}")

print("\nЗміщення стоїть на місці — скільки даних не давай, поліном 5-го степеня")
print("лишиться поліномом 5-го степеня. Це і є сенс слова «систематична».")
print(f"\nА дисперсія × n майже не рухається: {variance_by_size[0] * board_sizes[0]:.0f} → "
      f"{variance_by_size[-1] * board_sizes[-1]:.0f}, тоді як сама n виросла в "
      f"{board_sizes[-1] // board_sizes[0]} разів.")
print("Тобто дисперсія падає приблизно як 1/n.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))

ax.loglog(board_sizes, variance_by_size, "o-", color="teal", lw=2.5, label="дисперсія")
ax.loglog(board_sizes, bias_by_size, "o-", color="crimson", lw=2.5, label="зміщення²")
# еталонний нахил 1/n, привʼязаний до першої точки
reference = variance_by_size[0] * board_sizes[0] / board_sizes
ax.loglog(board_sizes, reference, ls="--", color="grey", lw=2, label="еталон 1/n")

ax.set_xlabel("кількість оголошень у дошці, n")
ax.set_ylabel("внесок у помилку, грн² (лог. шкала)")
ax.set_title("Більше даних — менша дисперсія. Зміщення не рухається")
ax.legend()
ax.grid(alpha=.25, which="both")
plt.tight_layout()
plt.show()

print("Бірюзова крива йде вниз паралельно сірому еталону — це і є закон 1/n.")
print("Рожева лежить майже горизонтально: дані на зміщення не діють.")

## 12. Ті самі криві готовими інструментами

Усе, що ми рахували руками, у `scikit-learn` уже є двома функціями. Писати руками
було потрібно, щоб зрозуміти, **що саме** вони рахують, — тепер можна користуватися.

- `validation_curve` — помилка від **складності** (наша U-подібна крива);
- `learning_curve` — помилка від **розміру вибірки** (та сама крива навчання,
  яку лекція радить будувати перед тим, як замовляти нові дані).

Обидві всередині роблять крос-валідацію, тому дають чесніші числа, ніж одне
відкладене розбиття. Візьмемо дошку на 150 оголошень, щоб кривій навчання було
куди рости.

In [ ]:
from sklearn.model_selection import validation_curve, learning_curve, KFold
from sklearn.preprocessing import StandardScaler

curve_rng = np.random.default_rng(11)
curve_years, curve_prices = draw_board(curve_rng, 150)
curve_x = to_unit(curve_years).reshape(-1, 1)      # sklearn чекає матрицю, а не вектор

# у конвеєрі степінь стоїть окремим кроком — саме його ми й будемо крутити
polynomial_model = make_pipeline(PolynomialFeatures(), StandardScaler(), LinearRegression())
folds = KFold(n_splits=5, shuffle=True, random_state=0)

degree_range = np.arange(1, 19)
train_scores, test_scores = validation_curve(
    polynomial_model, curve_x, curve_prices,
    param_name="polynomialfeatures__degree", param_range=degree_range,
    cv=folds, scoring="neg_root_mean_squared_error")

# sklearn повертає «чим більше, тим краще», тому міняємо знак назад
train_rmse = -train_scores.mean(axis=1)
test_rmse = -test_scores.mean(axis=1)
best_by_library = degree_range[test_rmse.argmin()]

print(f"{'степінь':>8} {'train':>10} {'test (CV)':>12}")
for i, degree in enumerate(degree_range):
    mark = "  ← мінімум" if degree == best_by_library else ""
    print(f"{degree:>8} {train_rmse[i]:>7.0f} грн {test_rmse[i]:>9.0f} грн{mark}")

print(f"\nvalidation_curve на дошці зі 150 оголошень каже: степінь {best_by_library}.")
print(f"Наш чесний експеримент із {N_BOARDS} паралельними дошками по 16 оголошень")
print(f"казав: степінь {best_by_decomposition}. Більша дошка дозволяє складнішу модель —")
print("рівно те, що ми бачили в розділі 5.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

axes[0].plot(degree_range, train_rmse, "o-", color="crimson", lw=2.5, label="train")
axes[0].plot(degree_range, test_rmse, "o-", color="teal", lw=2.5, label="test (CV)")
axes[0].axvline(best_by_library, ls="--", color="grey", label=f"мінімум: {best_by_library}")
axes[0].set_xlabel("степінь полінома")
axes[0].set_title("validation_curve: помилка від складності")

# крива навчання для замалої і для доречної моделі
for degree, colour in [(1, "crimson"), (5, "teal")]:
    train_sizes, _, holdout_scores = learning_curve(
        make_pipeline(PolynomialFeatures(degree), StandardScaler(), LinearRegression()),
        curve_x, curve_prices, train_sizes=np.linspace(0.2, 1.0, 7),
        cv=folds, scoring="neg_root_mean_squared_error")
    axes[1].plot(train_sizes, -holdout_scores.mean(axis=1), "o-", color=colour, lw=2.5,
                 label=f"степінь {degree}")

axes[1].axhline(PRICE_NOISE, ls=":", color="grey", label="підлога 750 грн")
axes[1].set_xlabel("кількість навчальних оголошень")
axes[1].set_title("learning_curve: помилка від кількості даних")

for ax in axes:
    ax.set_ylabel("RMSE, грн")
    ax.legend()
    ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

print("Читаємо праву картинку так, як радить лекція:")
print("  степінь 1 — крива вийшла на плато високо над підлогою. Дані вже не")
print("              допоможуть, проблема в моделі. Замовляти розмітку — марно.")
print("  степінь 5 — крива тисне до підлоги 750 грн. Ось тут нові дані")
print("              справді куплять тобі якість.")

## 13. Уся діагностика в одній таблиці

In [ ]:
print("що бачу                                    | діагноз       | що робити")
print("-" * 92)
print("обидві високі, розриву майже немає         | недонавчання  | ускладнити модель, додати ознак")
print("обидві низькі, розрив невеликий            | усе гаразд    | не чіпати")
print("навчальна дуже низька, тестова помітно вища| перенавчання  | спростити, додати даних, регуляризація")
print()
print("а якщо треба знати, ЧОМУ саме так:")
print("  вузьке віяло, середня крива мимо істини  | зміщення      | ускладнити")
print("  широке віяло, середня крива по істині    | дисперсія     | спростити або добути дані")

---

## 💻 Завдання

### 🟢 Рівень 1 — База

1. Заміни `PRICE_NOISE` із 750 на 200 і перебудуй таблицю з розділу 3.
   Назви новий найкращий степінь і поясни одним реченням, чому при меншому шумі
   вигідно брати складнішу модель.
2. Перебудуй розділ 9 із тим самим меншим шумом. Який доданок змінився, а які
   лишились на місці? Куди зсунувся мінімум суми?

**Зроблено, якщо:** обидва найкращі степені названі й ти пояснив(ла), чому вони
зсунулись в один бік.

### 🟡 Рівень 2 — Плюс

1. Заміни поліном на **k-NN** (`sklearn.neighbors.KNeighborsRegressor`) і побудуй
   ту саму таблицю зміщення/дисперсії, але по $k$ від 1 до 15. Функція
   `run_parallel_boards` майже не зміниться — треба лише інакше навчати модель.
2. Окремо перевір крайні випадки: $k = 1$ і $k = n$ (усі оголошення в сусідах).

**Зроблено, якщо:** видно **дзеркальну** картину до полінома — при $k=1$ дисперсія
максимальна, а зміщення мінімальне; при $k = n$ модель вироджується в константу
(нульова дисперсія, величезне зміщення). І ти написав(ла) одним реченням, що саме
усереднює k-NN, коли $k$ росте.

### 🔴 Рівень 3 — Виклик

Вибір степеня по тестовій вибірці — це підглядання у відповідь (розділ 09 лекції).
Поміряй, скільки коштує це підглядання.

1. Розбий дошку на три частини: навчальну, валідаційну й тестову.
2. Обери найкращий степінь **за валідаційною** частиною.
3. Поміряй помилку обраної моделі на валідаційній і на тестовій частинах.
4. Повтори кроки 1–3 щонайменше 30 разів із різними розбиттями й усередни обидва числа.

**Зроблено, якщо:** ти отримав(ла) дві середні величини — помилку на валідації
та на тесті — показав(ла), що перша систематично менша, назвав(ла) цю різницю
числом і пояснив(ла), звідки вона береться.

### Підказки

- Для рівня 1 достатньо перезапустити зошит згори з іншою константою; але
  памʼятай, що `rng` треба створити наново, інакше числа поїдуть.
- Для рівня 2 у k-NN «складність» крутиться в **протилежний** бік: маленьке $k$ —
  складна модель. Тому й таблиця вийде дзеркальною.
- Для рівня 3 не пиши цикл із нуля: `sklearn.model_selection.train_test_split`
  з різним `random_state` розбиває дані заново кожного разу.